# YOLO Sensitivity Analysis

Three analyses on the **YOLO pollinator detector** (`yolo_best.pt`), evaluated on the independent ground-truth eval set.

**Input** — `data/evaluation/images/` · `data/evaluation/annotations/` · `models/yolo_best.pt`  
**Output** — `outputs/evaluation/yolo_sensitivity_{timestamp}/` containing:

| File | Contents |
|------|----------|
| `yolo_robustness.png` | Fly recall under blur / darkness / JPEG compression |
| `yolo_ece.png` | Expected calibration error curve |
| `yolo_occlusion.png` | Occlusion sensitivity heatmap (bottom strip excluded) |
| `yolo_eigencam.png` | EigenCAM activation saliency overlays |

---

**How to run:**

1. *(Colab only)* Run **Cell 1** — mounts Drive and extracts the zip
2. Run **Cell 2** — imports, no edits needed
3. Edit and run **Cell 3** — the only cell you may need to change:
   - `YOLO_CKPT` — path to model weights (default: `models/yolo_best.pt`)
   - `EVAL_ANNOT` / `EVAL_IMAGES` — point to your annotations and images
   - `CONF` — detection confidence threshold (default: `0.2`)
4. Run **Cell 4** — loads YOLO and GT annotations (prints image counts)
5. Run **Cell 5** — robustness analysis (blur/darkness/JPEG; saves `yolo_robustness.png`)
6. Run **Cell 6** — ECE calibration curve (saves `yolo_ece.png`)
7. Run **Cell 7** — occlusion sensitivity scan (saves `yolo_occlusion.png`; slowest cell ~5 min)
8. Run **Cell 8** — EigenCAM saliency maps (saves `yolo_eigencam.png`)
9. Run **Cell 9** — prints output directory and lists saved files

> **Note:** All analyses use full-resolution images without SAHI tiling, to keep gradient and occlusion analysis tractable. The bottom `STRIP_H=120` px (camera OSD bar) is excluded from both detection and occlusion scanning.

YOLO classes: `fly=0 · butterfly=1 · other=2` (bumblebee not in YOLO training)  
GT annotation classes: `bumblebee=0 · fly=1 · butterfly=2 · other=3`

| Analysis | What it shows |
|---|---|
| **Robustness** | Fly recall under blur / darkness / JPEG compression |
| **ECE** | Whether YOLO confidence scores are well-calibrated |
| **Occlusion sensitivity** | Which image regions drive fly detection confidence |
| **EigenCAM** | Backbone activation map — which parts of the image the network attends to |

##### Cell 1 — Environment  *(no edits needed)*

In [13]:
import os
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('OMP_NUM_THREADS', '1')

import sys
import subprocess
from pathlib import Path

_git_root = Path(subprocess.check_output(
    ['git', 'rev-parse', '--show-toplevel'], text=True).strip())
BASE_DIR  = _git_root / 'ml_pipelines' / 'notebooks' / 'pollinator_detection'

MODEL_DIR   = BASE_DIR / 'models'
EVAL_ANNOT  = BASE_DIR / 'data' / 'evaluation' / 'annotations'
EVAL_IMAGES = BASE_DIR / 'data' / 'evaluation' / 'images'

print(f'BASE_DIR   : {BASE_DIR}  exists={BASE_DIR.exists()}')
print(f'MODEL_DIR  : {MODEL_DIR}  exists={MODEL_DIR.exists()}')
print(f'EVAL_ANNOT : {EVAL_ANNOT}  exists={EVAL_ANNOT.exists()}')
print(f'EVAL_IMAGES: {EVAL_IMAGES}  exists={EVAL_IMAGES.exists()}')

BASE_DIR   : /Users/lianshi/Downloads/bachelor thesis/automated-ecological-image-analysis/ml_pipelines/notebooks/pollinator_detection  exists=True
MODEL_DIR  : /Users/lianshi/Downloads/bachelor thesis/automated-ecological-image-analysis/ml_pipelines/notebooks/pollinator_detection/models  exists=True
EVAL_ANNOT : /Users/lianshi/Downloads/bachelor thesis/automated-ecological-image-analysis/ml_pipelines/notebooks/pollinator_detection/data/evaluation/annotations  exists=True
EVAL_IMAGES: /Users/lianshi/Downloads/bachelor thesis/automated-ecological-image-analysis/ml_pipelines/notebooks/pollinator_detection/data/evaluation/images  exists=True


##### Cell 2 — Imports

In [14]:
from io import BytesIO
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import torchvision.transforms.functional as TF
from PIL import Image
from ultralytics import YOLO

print('Imports OK')

Imports OK


##### Cell 3 — Config

In [15]:
YOLO_CKPT   = MODEL_DIR / 'yolo_best.pt'

# YOLO inference settings (same as production run)
CONF        = 0.05
NMS_IOU     = 0.45
STRIP_H     = 120    # bottom strip excluded (camera housing)

# YOLO class indices
YOLO_FLY    = 0

# Ground-truth annotation class indices
GT_FLY      = 1

# Evaluation settings
IOU_THRESH  = 0.50   # IoU threshold to count a detection as TP
SEED        = 42

# Occlusion settings
OCC_PATCH   = 75     # patch size in px
OCC_STRIDE  = 75     # stride (non-overlapping for speed)
OCC_N_IMGS  = 5      # number of images to visualise

np.random.seed(SEED)

# ── Output directory ────────────────────────────────────────────────────────
import datetime as _dt
_ts    = _dt.datetime.now().strftime('%Y%m%d_%H%M%S')
OUT_DIR = BASE_DIR / 'outputs' / 'evaluation' / f'yolo_sensitivity_{_ts}'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output dir → {OUT_DIR}')


Output dir → /Users/lianshi/Downloads/bachelor thesis/automated-ecological-image-analysis/ml_pipelines/notebooks/pollinator_detection/outputs/evaluation/yolo_sensitivity_20260528_233930


##### Cell 4 — Load model and GT annotations

In [16]:
# ── YOLO model ─────────────────────────────────────────────────────────────
yolo = YOLO(str(YOLO_CKPT))
print(f'Loaded: {YOLO_CKPT.name}')

# ── Load all GT annotations ────────────────────────────────────────────────
def load_gt(annot_root, image_root, gt_class_id):
    """
    Returns list of dicts:
        {'img_path': Path, 'boxes': [[x1,y1,x2,y2], ...]}
    Only images that contain at least one annotation for gt_class_id are included.
    Boxes are in pixel coordinates (xyxy).
    """
    records = []
    for scene_dir in sorted(annot_root.iterdir()):
        obj_dir = scene_dir / 'obj_train_data'
        img_dir = image_root / scene_dir.name
        if not obj_dir.exists() or not img_dir.exists():
            continue
        for txt_file in sorted(obj_dir.glob('*.txt')):
            stem = txt_file.stem
            img_path = None
            for ext in ('.JPG', '.jpg', '.jpeg', '.png', '.PNG'):
                c = img_dir / (stem + ext)
                if c.exists():
                    img_path = c
                    break
            if img_path is None:
                continue
            # Parse all annotations for this image
            all_boxes_by_cls = {}   # cls -> list of [x1,y1,x2,y2]
            img = Image.open(img_path)
            W, H = img.size
            img.close()
            for line in txt_file.read_text().strip().splitlines():
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                cls = int(parts[0])
                cx, cy, bw, bh = map(float, parts[1:])
                x1 = max(0, (cx - bw/2) * W)
                y1 = max(0, (cy - bh/2) * H)
                x2 = min(W, (cx + bw/2) * W)
                y2 = min(H, (cy + bh/2) * H)
                all_boxes_by_cls.setdefault(cls, []).append([x1, y1, x2, y2])
            if gt_class_id in all_boxes_by_cls:
                records.append({
                    'img_path': img_path,
                    'gt_boxes': all_boxes_by_cls[gt_class_id],
                    'img_size': (W, H),
                })
    return records

fly_records = load_gt(EVAL_ANNOT, EVAL_IMAGES, GT_FLY)
print(f'Images with ≥1 GT fly annotation: {len(fly_records)}')
print(f'Total GT fly boxes: {sum(len(r["gt_boxes"]) for r in fly_records)}')

# ── Helper: IoU between two [x1,y1,x2,y2] boxes ───────────────────────────
def box_iou(a, b):
    xi1 = max(a[0], b[0]);  yi1 = max(a[1], b[1])
    xi2 = min(a[2], b[2]);  yi2 = min(a[3], b[3])
    inter = max(0, xi2-xi1) * max(0, yi2-yi1)
    area_a = (a[2]-a[0]) * (a[3]-a[1])
    area_b = (b[2]-b[0]) * (b[3]-b[1])
    return inter / max(area_a + area_b - inter, 1e-8)

# ── Helper: run YOLO on a PIL image (no SAHI), return fly predictions ──────
def predict_fly(pil_img):
    """
    Returns list of dicts: {'box': [x1,y1,x2,y2], 'conf': float}
    Excludes detections in the bottom STRIP_H px (camera housing strip).
    """
    _H = pil_img.size[1]
    results = yolo.predict(pil_img, conf=CONF, iou=NMS_IOU, verbose=False)[0]
    preds = []
    for box, cls, conf in zip(
        results.boxes.xyxy.cpu().numpy(),
        results.boxes.cls.cpu().numpy().astype(int),
        results.boxes.conf.cpu().numpy()
    ):
        if cls == YOLO_FLY and box[1] < _H - STRIP_H:  # exclude bottom housing strip
            preds.append({'box': box.tolist(), 'conf': float(conf)})
    return preds

# ── Helper: compute recall for a list of records ───────────────────────────
def compute_fly_recall(records, distort_fn=None):
    """
    For each record, load image (optionally apply distort_fn: PIL->PIL),
    run YOLO, match predictions to GT boxes with greedy IoU matching.
    Returns (recall, n_tp, n_gt).
    """
    n_tp = 0; n_gt = 0
    for rec in records:
        img = Image.open(rec['img_path']).convert('RGB')
        if distort_fn is not None:
            img = distort_fn(img)
        preds  = predict_fly(img)
        gt_boxes = rec['gt_boxes']
        n_gt += len(gt_boxes)
        matched_gt = set()
        for pred in sorted(preds, key=lambda x: -x['conf']):
            best_iou, best_j = 0, -1
            for j, gt in enumerate(gt_boxes):
                if j in matched_gt:
                    continue
                iou = box_iou(pred['box'], gt)
                if iou > best_iou:
                    best_iou, best_j = iou, j
            if best_iou >= IOU_THRESH:
                n_tp += 1
                matched_gt.add(best_j)
    recall = n_tp / max(1, n_gt)
    return recall, n_tp, n_gt

Loaded: yolo_best.pt
Images with ≥1 GT fly annotation: 427
Total GT fly boxes: 602


##### Cell 5 — Robustness analysis

For each degradation type (Gaussian blur, brightness reduction, JPEG compression), gradually applies the distortion and measures fly recall. Shows how robust the detector is to real-world image quality variation.

In [ ]:
def jpeg_compress(quality):
    def _fn(img):
        buf = BytesIO()
        img.save(buf, format='JPEG', quality=quality)
        buf.seek(0)
        return Image.open(buf).copy()
    return _fn

distortions = [
    ('Baseline',      None),
    ('Blur σ=1',      lambda img: img.filter(__import__('PIL.ImageFilter', fromlist=['GaussianBlur']).GaussianBlur(radius=1))),
    ('Blur σ=2',      lambda img: img.filter(__import__('PIL.ImageFilter', fromlist=['GaussianBlur']).GaussianBlur(radius=2))),
    ('Blur σ=4',      lambda img: img.filter(__import__('PIL.ImageFilter', fromlist=['GaussianBlur']).GaussianBlur(radius=4))),
    ('Dark ×0.7',     lambda img: TF.adjust_brightness(img, 0.7)),
    ('Dark ×0.5',     lambda img: TF.adjust_brightness(img, 0.5)),
    ('Dark ×0.3',     lambda img: TF.adjust_brightness(img, 0.3)),
    ('JPEG q=75',     jpeg_compress(75)),
    ('JPEG q=50',     jpeg_compress(50)),
    ('JPEG q=20',     jpeg_compress(20)),
]

print(f'Running robustness on {len(fly_records)} images, {sum(len(r["gt_boxes"]) for r in fly_records)} GT fly boxes ...')
rob_results = {}
for label, fn in distortions:
    recall, tp, gt = compute_fly_recall(fly_records, distort_fn=fn)
    rob_results[label] = recall
    print(f'  {label:<14}: recall={recall:.3f}  (TP={tp}/{gt})')

Running robustness on 427 images, 602 GT fly boxes ...
  Baseline      : recall=0.140  (TP=84/602)
  Blur σ=1      : recall=0.135  (TP=81/602)
  Blur σ=2      : recall=0.111  (TP=67/602)
  Blur σ=4      : recall=0.060  (TP=36/602)
  Dark ×0.7     : recall=0.141  (TP=85/602)
  Dark ×0.5     : recall=0.128  (TP=77/602)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
fig.suptitle('YOLO fly recall under distortions — independent eval set (IoU ≥ 0.5)', fontsize=11)

baseline = rob_results['Baseline']
groups = [
    ('Gaussian blur',    ['Baseline', 'Blur σ=1', 'Blur σ=2', 'Blur σ=4'],
                         ['clean', 'σ=1', 'σ=2', 'σ=4']),
    ('Darkness',         ['Baseline', 'Dark ×0.7', 'Dark ×0.5', 'Dark ×0.3'],
                         ['clean', '×0.7', '×0.5', '×0.3']),
    ('JPEG compression', ['Baseline', 'JPEG q=75', 'JPEG q=50', 'JPEG q=20'],
                         ['clean', 'q=75', 'q=50', 'q=20']),
]

for ax, (title, keys, xticks) in zip(axes, groups):
    vals   = [rob_results[k] for k in keys]
    colors = ['#4C72B0' if i == 0 else '#DD8452' for i in range(len(keys))]
    bars   = ax.bar(xticks, vals, color=colors, edgecolor='white')
    ax.set_title(title, fontsize=10)
    ax.set_ylim(0, 1.05)
    if ax is axes[0]:
        ax.set_ylabel('Fly recall (IoU ≥ 0.5)', fontsize=10)
    ax.axhline(baseline, color='grey', lw=0.8, ls='--')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.01,
                f'{v:.2f}', ha='center', va='bottom', fontsize=8)
    ax.tick_params(axis='x', labelsize=8)

plt.tight_layout()
out = OUT_DIR / 'yolo_robustness.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')

##### Cell 6 — Expected Calibration Error (ECE)

Measures whether YOLO confidence scores are well-calibrated: a model that says 80 % should be right ~80 % of the time. Plots reliability diagram and computes ECE score.

In [ ]:
all_confs   = []   # confidence of each fly prediction
all_correct = []   # 1 if TP (matched GT), 0 if FP

print('Running ECE inference ...')
for rec in fly_records:
    img   = Image.open(rec['img_path']).convert('RGB')
    preds = predict_fly(img)
    gt_boxes = rec['gt_boxes']
    matched_gt = set()
    # Greedy matching (high confidence first)
    for pred in sorted(preds, key=lambda x: -x['conf']):
        best_iou, best_j = 0, -1
        for j, gt in enumerate(gt_boxes):
            if j in matched_gt:
                continue
            iou = box_iou(pred['box'], gt)
            if iou > best_iou:
                best_iou, best_j = iou, j
        is_tp = best_iou >= IOU_THRESH
        if is_tp:
            matched_gt.add(best_j)
        all_confs.append(pred['conf'])
        all_correct.append(float(is_tp))

all_confs   = np.array(all_confs)
all_correct = np.array(all_correct)

print(f'Total fly predictions: {len(all_confs)}')
print(f'TP (precision): {all_correct.mean():.3f}')

N_BINS = 10
bins     = np.linspace(0, 1, N_BINS + 1)
bin_acc  = np.zeros(N_BINS)
bin_conf = np.zeros(N_BINS)
bin_cnt  = np.zeros(N_BINS, dtype=int)

for i in range(N_BINS):
    mask = (all_confs >= bins[i]) & (all_confs < bins[i+1])
    if mask.sum() > 0:
        bin_acc[i]  = all_correct[mask].mean()
        bin_conf[i] = all_confs[mask].mean()
        bin_cnt[i]  = mask.sum()

ece = np.sum(np.abs(bin_acc - bin_conf) * bin_cnt) / len(all_confs)
print(f'ECE = {ece:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
bin_centres = (bins[:-1] + bins[1:]) / 2

ax.bar(bin_centres, bin_acc, width=0.08, alpha=0.85,
       label='Precision (TP rate)', color='#4C72B0', edgecolor='white')
ax.bar(bin_centres, np.maximum(bin_conf - bin_acc, 0), width=0.08,
       bottom=bin_acc, alpha=0.4, color='#DD8452',
       edgecolor='white', label='Overconfidence gap')
ax.plot([0, 1], [0, 1], 'k--', lw=1.2, label='Perfect calibration')

ax.set_xlabel('Confidence', fontsize=11)
ax.set_ylabel('Precision (TP rate)', fontsize=11)
ax.set_title(
    f'YOLO reliability diagram — fly detections, independent eval set\n'
    f'ECE = {ece:.4f}  |  n predictions = {len(all_confs)}',
    fontsize=10
)
ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)
ax.legend(fontsize=9); ax.grid(alpha=0.3)

for c, cnt in zip(bin_centres, bin_cnt):
    if cnt > 0:
        ax.text(c, 0.02, str(int(cnt)), ha='center', va='bottom',
                fontsize=7, color='white')

plt.tight_layout()
out = OUT_DIR / 'yolo_ece.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')

##### Cell 7 — Occlusion sensitivity

Slides a grey patch across the image and measures the drop in fly detection score. Bright areas in the heatmap = regions critical for detection. Bottom `STRIP_H` px (camera OSD bar) are excluded.

In [ ]:
import PIL.Image as PilImg

def total_fly_conf(pil_img):
    """Sum of all fly detection confidence scores in the image."""
    preds = predict_fly(pil_img)
    return sum(p['conf'] for p in preds)

def occlusion_heatmap(pil_img, patch_size=75, stride=75):
    """
    Returns a 2-D numpy array (n_rows × n_cols) of confidence-drop values.
    Higher value = masking this region hurts detection more = more important.
    """
    W, H = pil_img.size
    baseline_conf = total_fly_conf(pil_img)

    xs = list(range(0, W - patch_size + 1, stride))
    ys = list(range(0, H - STRIP_H - patch_size + 1, stride))   # skip bottom camera-housing strip

    heatmap = np.zeros((len(ys), len(xs)))

    for ri, y in enumerate(ys):
        for ci, x in enumerate(xs):
            masked = pil_img.copy()
            # Fill patch with mid-grey (128)
            patch = PilImg.new('RGB', (patch_size, patch_size), (128, 128, 128))
            masked.paste(patch, (x, y))
            drop = baseline_conf - total_fly_conf(masked)
            heatmap[ri, ci] = max(0, drop)   # only show drops, not gains

    return heatmap, xs, ys, baseline_conf

# Pick images with most GT fly boxes for interesting heatmaps
sorted_records = sorted(fly_records, key=lambda r: -len(r['gt_boxes']))
sample_records = sorted_records[:OCC_N_IMGS]

fig, axes = plt.subplots(OCC_N_IMGS, 2,
                          figsize=(12, 4.5 * OCC_N_IMGS))
fig.suptitle('YOLO occlusion sensitivity — fly detection confidence drop\n'
             '(red = high confidence drop = important region)', fontsize=12, y=1.01)

for row, rec in enumerate(sample_records):
    img = Image.open(rec['img_path']).convert('RGB')
    W, H = img.size
    print(f'[{row+1}/{OCC_N_IMGS}] {rec["img_path"].name}  '
          f'({W}×{H}, {len(rec["gt_boxes"])} GT flies) ...')

    hm, xs, ys, base_conf = occlusion_heatmap(img, OCC_PATCH, OCC_STRIDE)
    print(f'  baseline_conf={base_conf:.3f}  max_drop={hm.max():.3f}')

    # Upsample heatmap to image size for overlay
    hm_img = PilImg.fromarray((hm / max(hm.max(), 1e-8) * 255).astype('uint8'))
    # Place heatmap cells at their correct pixel positions
    hm_full = np.zeros((H, W))
    cell_h = OCC_STRIDE; cell_w = OCC_STRIDE
    for ri, y in enumerate(ys):
        for ci, x in enumerate(xs):
            val = hm[ri, ci] / max(hm.max(), 1e-8)
            hm_full[y:y+cell_h, x:x+cell_w] = val

    img_np = np.array(img)
    # Use Reds: white = no drop, red = large drop.
    # Alpha blending: important regions (high drop) show strong red,
    # unimportant regions stay transparent so the original image shows through.
    heat_rgba  = cm.Reds(hm_full)          # (H, W, 4)
    alpha_mask = hm_full[:, :, np.newaxis] # 0 = fully transparent
    overlay    = np.clip(
        img_np / 255.0 * (1 - 0.65 * alpha_mask) + heat_rgba[:, :, :3] * 0.65 * alpha_mask,
        0, 1
    )

    # Draw GT boxes on original
    import matplotlib.patches as patches
    ax_orig = axes[row, 0]
    ax_orig.imshow(img_np)
    for gt in rec['gt_boxes']:
        rect = patches.Rectangle(
            (gt[0], gt[1]), gt[2]-gt[0], gt[3]-gt[1],
            linewidth=1.5, edgecolor='lime', facecolor='none'
        )
        ax_orig.add_patch(rect)
    ax_orig.set_title(f'{rec["img_path"].name}  ({len(rec["gt_boxes"])} flies)',
                      fontsize=8)
    ax_orig.axis('off')

    ax_heat = axes[row, 1]
    ax_heat.imshow(overlay)
    ax_heat.set_title(f'Occlusion heatmap  (baseline conf={base_conf:.2f})', fontsize=8)
    ax_heat.axis('off')

plt.tight_layout()
out = OUT_DIR / 'yolo_occlusion.png'
plt.savefig(out, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')

##### Cell 8 — EigenCAM saliency

Applies EigenCAM to YOLO's last backbone conv layer. Highlights which image regions activate the network most strongly. No backprop required — uses SVD on activations only.

In [ ]:
# ── EigenCAM on YOLO ─────────────────────────────────────────────────────────
try:
    from pytorch_grad_cam import EigenCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'grad-cam', '-q'])
    from pytorch_grad_cam import EigenCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

import torch
import torchvision.transforms.functional as TF
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib; matplotlib.use('Agg')

EC_N_IMGS = 5

_sorted_ec  = sorted(fly_records, key=lambda r: -len(r['gt_boxes']))
_ec_records = _sorted_ec[:EC_N_IMGS]

# Target layer: last backbone conv before the Detect head
_target_layers = [yolo.model.model[-2]]

# EigenCAM uses activations only — no backprop needed.
# yolo.model.train() makes forward return raw tensors (not Results objects).
# YOLO returns a tuple in training mode; we pass explicit targets to EigenCAM
# so it skips the np.argmax step (EigenCAM only needs layer activations, not outputs).
yolo.model.train()   # training mode → returns tensors, not Results objects
_use_cuda = torch.cuda.is_available()
_cam = EigenCAM(yolo.model, _target_layers)  # device inferred from model

fig, axes = plt.subplots(EC_N_IMGS, 2, figsize=(12, 4.5 * EC_N_IMGS))
fig.suptitle('YOLO EigenCAM — activation saliency map\n'
             '(warm colour = region important for fly detection)',
             fontsize=12, y=1.01)

_INPUT_SIZE = 640

for _row, _rec in enumerate(_ec_records):
    _img_pil = Image.open(_rec['img_path']).convert('RGB')
    _img_np  = np.array(_img_pil) / 255.0

    _img_resized = _img_pil.resize((_INPUT_SIZE, _INPUT_SIZE))
    _img_res_np  = np.array(_img_resized) / 255.0
    _tensor = TF.to_tensor(_img_resized).unsqueeze(0).float()

    # YOLO returns a tuple in train mode — pass a dummy target to skip argmax
    _targets   = [ClassifierOutputTarget(0)]  # EigenCAM ignores targets; needed to skip argmax
    _grayscale = _cam(input_tensor=_tensor, targets=_targets)[0]
    _overlay   = show_cam_on_image(_img_res_np, _grayscale, use_rgb=True)

    # Left: original + GT boxes
    _ax_orig = axes[_row, 0]
    _ax_orig.imshow(np.array(_img_pil))
    for _gt in _rec['gt_boxes']:
        _rect = patches.Rectangle(
            (_gt[0], _gt[1]), _gt[2]-_gt[0], _gt[3]-_gt[1],
            linewidth=1.5, edgecolor='lime', facecolor='none')
        _ax_orig.add_patch(_rect)
    _ax_orig.set_title(f'{_rec["img_path"].name}  ({len(_rec["gt_boxes"])} flies)', fontsize=8)
    _ax_orig.axis('off')

    # Right: EigenCAM overlay
    _ax_cam = axes[_row, 1]
    _ax_cam.imshow(_overlay)
    _ax_cam.set_title('EigenCAM (640x640 input)', fontsize=8)
    _ax_cam.axis('off')

yolo.model.eval()   # restore eval mode
_cam.activations_and_grads.release()

plt.tight_layout()
_ec_out = OUT_DIR / 'yolo_eigencam.png'
plt.savefig(_ec_out, dpi=120, bbox_inches='tight')
plt.close(fig)
print(f'Saved -> {_ec_out}')


##### Cell 9 — Output summary

Prints the output directory path and lists all saved files.